# To Run Spectral Ratio Illumination Demo on Google Colab

Omar Elmady 

Wednesday, Dec 11 

CS 7180 

## Setup Steps:
1. **Enable GPU Runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. **Update model link** in Step 2 if you have it from your professor
3. **Run all cells in order** (Runtime → Run all)

## What this notebook does:
- Checks GPU availability
- Downloads model from Google Drive (if link provided)
- Clones your GitHub repository
- Installs all dependencies (PyTorch with GPU/CPU support)
- Runs validation checks
- Processes images with your algorithms
- Downloads results as a .tar.gz file

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
!nvidia-smi || echo "⚠️ No GPU detected - will use CPU (slower but works)"

## Step 2: Clone Repository and Download Model

### 2a. Clone Repository

First, clone the GitHub repository to get all the code and data.

In [ ]:
import os

# Remove if already exists from previous runs
if os.path.exists('/content/Spectral_Ratio_Illumination_Demo'):
    !rm -rf /content/Spectral_Ratio_Illumination_Demo

# Clone the repository
print("📥 Cloning repository from GitHub...")
!git clone https://github.com/oelmady/Spectral_Ratio_Illumination_Demo.git /content/Spectral_Ratio_Illumination_Demo

# Change to the project directory
%cd /content/Spectral_Ratio_Illumination_Demo

print("\n✓ Repository cloned successfully!")
print("\n📁 Repository contents:")
!ls -lh

### 2b. Download Model from Google Drive

To extract file ID from a Drive link like `https://drive.google.com/file/d/1ABC123XYZ/view?usp=sharing`, copy the `1ABC123XYZ` part. If you need to change it, update `MODEL_DRIVE_ID` below.  

The model will download directly into the repository's `model/` folder.

In [ ]:
import os

# ============================================
# CONFIGURATION: Google Drive file ID for model
# ============================================
MODEL_DRIVE_ID = "1h2fVtLQJpgLl4_C3MLA_VDuqlJTcAqf6"
# ============================================

if MODEL_DRIVE_ID:
    print("📥 Downloading model from Google Drive...")
    
    # Install gdown for Drive downloads
    !pip install -q gdown
    
    import gdown
    
    # Download directly into the repo's model/ directory
    model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'
    url = f'https://drive.google.com/uc?id={MODEL_DRIVE_ID}'
    
    try:
        gdown.download(url, model_path, quiet=False)
        
        # Verify download
        if os.path.exists(model_path):
            size_mb = os.path.getsize(model_path) / (1024 * 1024)
            if size_mb > 100:  # Should be ~528MB
                print(f"\n✓ Model downloaded successfully: {size_mb:.1f} MB")
                print(f"   Location: {model_path}")
            else:
                print(f"\n⚠️ Model file seems too small ({size_mb:.1f} MB)")
                print("   Check if the Drive link allows public access")
        else:
            print("\n❌ Model download failed")
            print("   Make sure the file is shared with 'Anyone with the link'")
    except Exception as e:
        print(f"\n❌ Error downloading model: {e}")
        print("   Double-check the file ID and sharing permissions")
else:
    print("⚠️ No model file ID provided")
    print("   Will run baseline-only experiments (no neural ISD prediction)")

# Show model directory contents
print("\n📁 Model directory:")
!ls -lh /content/Spectral_Ratio_Illumination_Demo/model/


## Step 3: Install Dependencies

This cell installs all required Python packages:
- PyTorch (GPU version if available, otherwise CPU)
- OpenCV (full version with GUI support)
- NumPy, Matplotlib, and other dependencies

In [ ]:
import subprocess
import sys

print("📦 Installing dependencies...\n")

# Upgrade pip
print("1️⃣ Upgrading pip...")
!pip install --quiet --upgrade pip setuptools wheel

# Install opencv (full version, not headless - Colab supports GUI)
print("\n2️⃣ Installing OpenCV, NumPy, Matplotlib...")
!pip install --quiet opencv-python numpy matplotlib

# Install PyTorch with GPU support if available
print("\n3️⃣ Installing PyTorch...")
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("   GPU detected: Installing CUDA-enabled PyTorch (cu118)")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
except:
    print("   No GPU: Installing CPU-only PyTorch")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install any remaining requirements
print("\n4️⃣ Installing remaining requirements...")
!pip install --quiet -r requirements.txt 2>/dev/null || true

print("\n✓ All dependencies installed successfully!")

# Verify installations
print("\n📋 Checking installed versions:")
import torch
import cv2
import numpy as np
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   OpenCV: {cv2.__version__}")
print(f"   NumPy: {np.__version__}")

## Step 4: Run Preflight Check

This validates that everything is set up correctly.

In [ ]:
%cd /content/Spectral_Ratio_Illumination_Demo
!python preflight_check.py

## Step 5: Run Experiments

This cell processes all images in `data/images/` with your algorithms.

The model was downloaded automatically in Step 2, so this will run the full pipeline:
- Neural ISD prediction
- SR-constrained Retinex
- Baseline Retinex (for comparison)
- SR-based color correction

In [ ]:
import os
import sys

%cd /content/Spectral_Ratio_Illumination_Demo

# Set PYTHONPATH environment variable so subprocess can find modules
os.environ['PYTHONPATH'] = '/content/Spectral_Ratio_Illumination_Demo'

model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'

if os.path.exists(model_path) and os.path.getsize(model_path) > 1000000:  # > 1MB
    print("🚀 Running FULL experiment (with neural ISD model)...")
    print("   This includes: model inference + SR-Retinex + baseline Retinex + color correction\n")
    !python scripts/run_batch.py \
        --use-model \
        --retinex \
        --baseline-retinex \
        --sr-correct \
        --iterations 5 \
        --sigma 15 \
        --distance 1.0
else:
    print("🚀 Running BASELINE experiment (no model)...")
    print("   This includes: baseline Retinex + SR-constrained Retinex (using annotated maps)\n")
    print("   ⚠️ Note: Since no model is available, this will use pre-existing SR maps")
    print("   from data/sr_maps/ if available, or skip SR-constrained processing.\n")
    !python scripts/run_batch.py \
        --baseline-retinex \
        --retinex \
        --iterations 5 \
        --sigma 15

print("\n✓ Processing complete! Check results/ directory.")


## Step 6: View Sample Results (Optional)

Display a few output images to verify processing worked correctly.

In [ ]:
import os
import glob
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Find PNG outputs in results directory
png_files = glob.glob('/content/Spectral_Ratio_Illumination_Demo/results/*.png')

if png_files:
    print(f"📸 Found {len(png_files)} output images. Showing first 3:\n")
    for img_path in png_files[:3]:
        print(f"   {os.path.basename(img_path)}")
        display(Image(filename=img_path, width=600))
        print()
else:
    print("⚠️ No PNG outputs found in results/")
    print("   Check if processing completed successfully above.")

## Step 7: Package and Download Results

This creates a `.tar.gz` archive of all results and downloads it to your local machine.

In [ ]:
import os
from google.colab import files

%cd /content/Spectral_Ratio_Illumination_Demo

# Determine what to package
has_results = os.path.exists('results')
has_tuning = os.path.exists('results_tuning')

if has_tuning:
    # If tuning was run, package all tuning results
    print("📦 Packaging parameter tuning results...")
    !tar -czf results_tuning.tar.gz results_tuning/ 2>/dev/null
    
    if os.path.exists('results_tuning.tar.gz'):
        size_mb = os.path.getsize('results_tuning.tar.gz') / (1024 * 1024)
        print(f"✓ Archive created: results_tuning.tar.gz ({size_mb:.1f} MB)")
        print("\n⬇️ Downloading to your computer...")
        files.download('results_tuning.tar.gz')
        print("✓ Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results_tuning.tar.gz")
    else:
        print("⚠️ Failed to create tuning archive")

elif has_results:
    # If only single run results exist, package those
    print("📦 Packaging results...")
    !tar -czf results.tar.gz results/ 2>/dev/null
    
    if os.path.exists('results.tar.gz'):
        size_mb = os.path.getsize('results.tar.gz') / (1024 * 1024)
        print(f"✓ Archive created: results.tar.gz ({size_mb:.1f} MB)")
        print("\n⬇️ Downloading to your computer...")
        files.download('results.tar.gz')
        print("✓ Download complete!")
        print("\nTo extract on your Mac:")
        print("   tar -xzf results.tar.gz")
    else:
        print("⚠️ Failed to create results archive")

else:
    print("⚠️ No results to package. Make sure Step 5 or parameter tuning completed successfully.")

---

## Optional: Automated Parameter Tuning

Run systematic experiments with different parameter combinations to find optimal settings.

**What this does:**
- Tests multiple values for iterations, sigma (blur), and distance (color correction)
- Saves results for each combination separately
- Helps you find the best parameters for your images

**Note**: This will take longer (proportional to number of parameter combinations × number of images)

In [ ]:
import os
import sys
import shutil
from datetime import datetime

%cd /content/Spectral_Ratio_Illumination_Demo

# Add current directory to Python path so imports work
sys.path.insert(0, '/content/Spectral_Ratio_Illumination_Demo')

# Define parameter ranges to test
iterations_values = [3, 5, 10]  # Number of Retinex iterations
sigma_values = [10, 15, 20, 25]  # Gaussian blur sigma (larger = more smoothing)
distance_values = [0.5, 1.0, 1.5]  # SR color correction distance

print("🔬 Starting Automated Parameter Tuning")
print(f"   Testing {len(iterations_values)} × {len(sigma_values)} × {len(distance_values)} = {len(iterations_values) * len(sigma_values) * len(distance_values)} combinations\n")

# Create a directory for tuning results
tuning_dir = 'results_tuning'
os.makedirs(tuning_dir, exist_ok=True)

total_runs = len(iterations_values) * len(sigma_values) * len(distance_values)
current_run = 0

# Test each combination
for iterations in iterations_values:
    for sigma in sigma_values:
        for distance in distance_values:
            current_run += 1
            print(f"\n{'='*70}")
            print(f"Run {current_run}/{total_runs}: iterations={iterations}, sigma={sigma}, distance={distance}")
            print('='*70)
            
            # Run the batch processor with these parameters
            !python scripts/run_batch.py \
                --use-model \
                --retinex \
                --baseline-retinex \
                --sr-correct \
                --iterations {iterations} \
                --sigma {sigma} \
                --distance {distance}
            
            # Move results to a labeled subdirectory
            result_subdir = f"{tuning_dir}/iter{iterations}_sigma{sigma}_dist{distance}"
            if os.path.exists('results'):
                if os.path.exists(result_subdir):
                    shutil.rmtree(result_subdir)
                shutil.copytree('results', result_subdir)
                print(f"✓ Results saved to: {result_subdir}")
            
            # Clean up results directory for next run
            if os.path.exists('results'):
                shutil.rmtree('results')

print(f"\n\n{'='*70}")
print("✓ Parameter tuning complete!")
print(f"   All results saved in: {tuning_dir}/")
print('='*70)

# Show summary of all runs
print("\n📊 Summary of parameter combinations tested:")
for iterations in iterations_values:
    for sigma in sigma_values:
        for distance in distance_values:
            result_subdir = f"{tuning_dir}/iter{iterations}_sigma{sigma}_dist{distance}"
            if os.path.exists(result_subdir):
                num_files = len([f for f in os.listdir(result_subdir) if f.endswith('.png')])
                print(f"   ✓ iter={iterations:2d}, sigma={sigma:2d}, dist={distance:.1f} → {num_files} output images")

print("\n💡 Next steps:")
print("   1. Review images in each subdirectory")
print("   2. Compare visual quality across parameter settings")
print("   3. Select the best combination for your final results")
print("   4. Run Step 7 below to download all tuning results")


## Optional: Visual Comparison for Presentation

**For your presentation, focus on these comparisons:**

1. **Method comparison** - Show one parameter setting with all 4 outputs:
   - Original input (8-bit reference)
   - Baseline Retinex (standard method)
   - SR-constrained Retinex (your contribution)
   - SR color correction (additional improvement)

2. **Parameter sensitivity** - Pick 2-3 parameter variations that show clear differences

**Expected Results to Highlight:**
- SR-constrained should preserve color better than baseline (less color shifts)
- Color correction should improve white balance/color cast issues
- Baseline may over-smooth or change colors unrealistically
- Your method should show better detail preservation in shadowed regions

The cell below will display **side-by-side method comparisons** from one parameter setting.


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

tuning_dir = 'results_tuning'

if not os.path.exists(tuning_dir):
    print("⚠️ No tuning results found. Run the parameter tuning cell above first.")
else:
    # Find all subdirectories (one per parameter combination)
    subdirs = sorted([d for d in os.listdir(tuning_dir) if os.path.isdir(os.path.join(tuning_dir, d))])
    
    if not subdirs:
        print("⚠️ No result subdirectories found in results_tuning/")
    else:
        print(f"📊 Found {len(subdirs)} parameter combinations\n")
        
        # Pick a middle-ground parameter setting (iter5_sigma15_dist1.0)
        target_dir = 'iter5_sigma15_dist1.0'
        if target_dir in subdirs:
            sample_dir = os.path.join(tuning_dir, target_dir)
        else:
            # Fallback to first directory if target not found
            sample_dir = os.path.join(tuning_dir, subdirs[len(subdirs)//2])
            target_dir = subdirs[len(subdirs)//2]
        
        print(f"📸 Showing method comparison for: {target_dir}\n")
        
        # Find one image and get all its variants
        all_files = os.listdir(sample_dir)
        
        # Find a base image name (without suffix)
        base_names = set()
        for f in all_files:
            if f.endswith('_8bit.png'):
                base_names.add(f.replace('_8bit.png', ''))
        
        if base_names:
            base_name = sorted(base_names)[0]  # Pick first image
            
            # Define the comparison sequence
            comparisons = [
                ('_8bit.png', 'Original Input'),
                ('_baseline_retinex.png', 'Baseline Retinex'),
                ('_retinex.png', 'SR-Constrained Retinex'),
                ('_sr_corrected.png', 'SR Color Correction')
            ]
            
            # Load images
            images_to_show = []
            labels_to_show = []
            
            for suffix, label in comparisons:
                img_path = os.path.join(sample_dir, base_name + suffix)
                if os.path.exists(img_path):
                    images_to_show.append(img_path)
                    labels_to_show.append(label)
            
            if images_to_show:
                # Display in 2x2 grid
                fig, axes = plt.subplots(2, 2, figsize=(14, 14))
                axes = axes.flatten()
                
                for idx, (img_path, label) in enumerate(zip(images_to_show, labels_to_show)):
                    img = Image.open(img_path)
                    axes[idx].imshow(img)
                    axes[idx].set_title(label, fontsize=14, fontweight='bold')
                    axes[idx].axis('off')
                
                # Hide unused subplots
                for idx in range(len(images_to_show), 4):
                    axes[idx].axis('off')
                
                plt.tight_layout()
                plt.show()
                
                print(f"\n✅ Comparison for: {base_name}")
                print("\n💡 What to look for in your presentation:")
                print("   1. Color preservation: SR-constrained should maintain original colors better")
                print("   2. Detail recovery: Check shadow regions for detail preservation")
                print("   3. White balance: Color correction should fix color casts")
                print("   4. Artifacts: Baseline may introduce color shifts or halos")
                print("\n📝 Presentation tip: Use this comparison to show your method's advantages!")
                
            else:
                print("⚠️ Could not find complete set of output images")
        else:
            print("⚠️ No images found in selected parameter directory")


## Presentation Guidelines: What Results Demonstrate Value

**Key Findings to Present:**

### 1. **Problem Statement**
- Real-world images often have uneven illumination
- Standard methods (baseline Retinex) can introduce color distortions
- Goal: Illuminate images while preserving accurate colors

### 2. **Your Solution**
- **SR-Constrained Retinex**: Constrains illumination updates to spectral-ratio directions
- Preserves color relationships better than baseline
- Uses neural network to predict per-pixel illumination directions

### 3. **Quantitative Metrics** (if you calculate them):
- Color constancy error (how much colors shift)
- Detail preservation (SSIM in shadow regions)
- Computational time comparison

### 4. **Qualitative Results to Show**:
- **Best case**: Image where SR-constrained clearly outperforms baseline
  - Look for: better color accuracy, fewer artifacts
- **Color preservation**: Side-by-side showing baseline changes colors, yours doesn't
- **Detail recovery**: Zoom into shadow regions showing detail preservation

### 5. **Parameter Sensitivity** (optional):
- Show 2-3 sigma values: demonstrate you can control smoothness
- Higher sigma = smoother illumination (may lose detail)
- Lower sigma = preserves detail (may amplify noise)

**What Makes a Good Presentation Result:**
✅ Clear visual difference between methods  
✅ Your method solves a visible problem (color shift, over-smoothing)  
✅ Works on multiple images (show 2-3 examples)  
✅ Can explain WHY your method is better (spectral-ratio constraint)

**Avoid:**
❌ Showing all 36 parameter combinations  
❌ Results that look identical between methods  
❌ Only showing small differences that are hard to see
